### **CELULA 1: IMPORTACAO DE BIBLIOTECAS**

Nesta celula, importamos as bibliotecas necessarias para trabalhar com imagens medicas, extrair features e treinar os modelos SVM e Random Forest.

- `numpy`, `pandas`: manipulacao de dados
- `matplotlib`, `seaborn`: visualizacao
- `scikit-learn`: validacao cruzada, PCA, SVM, Random Forest e metricas
- `scikit-image` e `PyWavelets`: extracao de features (HOG, LBP, wavelets)

Se alguma dependencia nao estiver instalada, esta celula avisa como instalar.

In [ ]:
# ==============================================================================
# CELULA 1: IMPORTACAO DE BIBLIOTECAS
# ==============================================================================
import os
from pathlib import Path
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    roc_auc_score
)
from sklearn.pipeline import Pipeline
from sklearn.base import clone

sns.set(style="whitegrid")

# Dependencias para extracao de features
try:
    from skimage.feature import hog, local_binary_pattern
    skimage_ok = True
except Exception:
    skimage_ok = False
    print('AVISO: scikit-image nao encontrado. Instale com: pip install scikit-image')

try:
    import pywt
    pywt_ok = True
except Exception:
    pywt_ok = False
    print('AVISO: PyWavelets nao encontrado. Instale com: pip install PyWavelets')

print('Bibliotecas importadas com sucesso!')
print(f'scikit-image OK? {skimage_ok} | PyWavelets OK? {pywt_ok}')

### **CELULA 2: CONFIGURACAO DO DATASET E INSPECAO RAPIDA**

O dataset Chest X-Ray Pneumonia deve estar dentro de `data/`.
Estrutura esperada:

- `data/chest_xray/train/NORMAL`
- `data/chest_xray/train/PNEUMONIA`
- `data/chest_xray/test/NORMAL`
- `data/chest_xray/test/PNEUMONIA`

Nesta celula, contamos as imagens por classe e exibimos algumas amostras.

In [ ]:
# ==============================================================================
# CELULA 2: CONFIGURACAO DO DATASET E INSPECAO RAPIDA
# ==============================================================================
DATASET_DIR = Path('data/chest_xray')
SUBSETS = ['train', 'test']
CLASSES = ['NORMAL', 'PNEUMONIA']
IMG_EXTS = {'.png', '.jpg', '.jpeg'}
LABEL_MAP = {'NORMAL': 0, 'PNEUMONIA': 1}
RANDOM_STATE = 42

def listar_imagens(root_dir):
    paths = []
    labels = []
    for subset in SUBSETS:
        for class_name in CLASSES:
            class_dir = root_dir / subset / class_name
            if not class_dir.exists():
                continue
            for p in class_dir.rglob('*'):
                if p.suffix.lower() in IMG_EXTS:
                    paths.append(p)
                    labels.append(LABEL_MAP[class_name])
    return paths, np.array(labels)

paths, labels = listar_imagens(DATASET_DIR)
print(f'Total de imagens encontradas: {len(paths)}')

# Tabela de contagem por classe
if len(paths) > 0:
    df_counts = pd.DataFrame({'label': labels}).replace({0: 'NORMAL', 1: 'PNEUMONIA'})
    print(df_counts['label'].value_counts())
else:
    print('AVISO: nenhuma imagem encontrada. Verifique o caminho do dataset.')

# Visualizar algumas imagens de exemplo
if len(paths) > 0:
    n_show = 6
    idx = np.random.default_rng(RANDOM_STATE).choice(len(paths), size=min(n_show, len(paths)), replace=False)
    plt.figure(figsize=(10, 4))
    for i, k in enumerate(idx, 1):
        img = Image.open(paths[k]).convert('L')
        plt.subplot(2, 3, i)
        plt.imshow(img, cmap='gray')
        plt.title('NORMAL' if labels[k] == 0 else 'PNEUMONIA')
        plt.axis('off')
    plt.tight_layout()
    plt.show()

### **CELULA 3: PRE-PROCESSAMENTO E EXTRACAO DE FEATURES (HOG, LBP, WAVELETS)**

Nesta etapa, as imagens sao convertidas para escala de cinza, redimensionadas e transformadas em vetores de features.

- **HOG**: captura bordas e formas globais
- **LBP**: captura textura local
- **Wavelets**: capturam componentes de frequencia e textura multiescala

As features sao concatenadas em um unico vetor por imagem.

In [ ]:
# ==============================================================================
# CELULA 3: PRE-PROCESSAMENTO E EXTRACAO DE FEATURES
# ==============================================================================
IMG_SIZE = (128, 128)

def carregar_imagem(path, size=IMG_SIZE):
    img = Image.open(path).convert('L')
    img = img.resize(size)
    return np.array(img, dtype=np.float32)

def extrair_hog(img):
    if not skimage_ok:
        raise RuntimeError('scikit-image nao disponivel para HOG.')
    return hog(
        img,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm='L2-Hys',
        visualize=False,
        feature_vector=True
    )

def extrair_lbp(img, radius=3, n_points=24):
    if not skimage_ok:
        raise RuntimeError('scikit-image nao disponivel para LBP.')
    lbp = local_binary_pattern(img, n_points, radius, method='uniform')
    n_bins = n_points + 2
    hist, _ = np.histogram(lbp.ravel(), bins=np.arange(0, n_bins + 1), range=(0, n_bins), density=True)
    return hist

def extrair_wavelet(img, wavelet='db1', level=2):
    if not pywt_ok:
        raise RuntimeError('PyWavelets nao disponivel para wavelets.')
    coeffs = pywt.wavedec2(img, wavelet=wavelet, level=level)
    feats = []
    cA = coeffs[0]
    feats.extend([cA.mean(), cA.std(), np.sum(cA ** 2)])
    for (cH, cV, cD) in coeffs[1:]:
        for c in (cH, cV, cD):
            feats.extend([c.mean(), c.std(), np.sum(c ** 2)])
    return np.array(feats, dtype=np.float32)

def extrair_features(img):
    hog_feat = extrair_hog(img)
    lbp_feat = extrair_lbp(img)
    wav_feat = extrair_wavelet(img)
    return np.concatenate([hog_feat, lbp_feat, wav_feat]).astype(np.float32)

### **CELULA 4: CONSTRUCAO DO DATASET DE FEATURES**

Aqui carregamos as imagens, extraimos as features e construimos a matriz X e o vetor y.

Para acelerar os testes, ajuste `MAX_POR_CLASSE` para limitar a quantidade por classe.

In [ ]:
# ==============================================================================
# CELULA 4: CONSTRUCAO DO DATASET DE FEATURES
# ==============================================================================
MAX_POR_CLASSE = 1500  # use None para carregar tudo

def limitar_por_classe(paths, labels, max_por_classe, seed=RANDOM_STATE):
    if max_por_classe is None:
        return paths, labels
    rng = np.random.default_rng(seed)
    paths = np.array(paths)
    labels = np.array(labels)
    idx_final = []
    for cls in np.unique(labels):
        idx_cls = np.where(labels == cls)[0]
        if len(idx_cls) > max_por_classe:
            idx_cls = rng.choice(idx_cls, size=max_por_classe, replace=False)
        idx_final.extend(idx_cls.tolist())
    idx_final = np.array(idx_final)
    return paths[idx_final].tolist(), labels[idx_final]

paths_sel, labels_sel = limitar_por_classe(paths, labels, MAX_POR_CLASSE)
print(f'Amostras usadas: {len(paths_sel)}')

X_list = []
y_list = []
start = time.perf_counter()
for i, (p, label) in enumerate(zip(paths_sel, labels_sel), 1):
    img = carregar_imagem(p)
    feats = extrair_features(img)
    X_list.append(feats)
    y_list.append(label)
    if i % 200 == 0:
        print(f'Processadas {i}/{len(paths_sel)} imagens...')

X = np.vstack(X_list)
y = np.array(y_list)
end = time.perf_counter()
print(f'X shape: {X.shape} | y shape: {y.shape}')
print(f'Tempo total de extracao: {end - start:.2f} s')

print(pd.Series(y).map({0: 'NORMAL', 1: 'PNEUMONIA'}).value_counts())

### **CELULA 5: REDUCAO DE DIMENSIONALIDADE (PCA + T-SNE)**

O PCA reduz a dimensionalidade mantendo a maior parte da variancia. Em seguida, o t-SNE e usado apenas para visualizacao 2D.

Esta etapa e exploratoria e nao substitui a reducao usada dentro da validacao cruzada.

In [ ]:
# ==============================================================================
# CELULA 5: REDUCAO DE DIMENSIONALIDADE (PCA + T-SNE)
# ==============================================================================
scaler_vis = StandardScaler()
X_scaled_vis = scaler_vis.fit_transform(X)

pca_vis = PCA(n_components=0.95, random_state=RANDOM_STATE)
X_pca_vis = pca_vis.fit_transform(X_scaled_vis)
print(f'PCA -> dimensoes: {X_pca_vis.shape[1]} (variancia explicada: {pca_vis.explained_variance_ratio_.sum():.3f})')

max_tsne = 1500
rng = np.random.default_rng(RANDOM_STATE)
idx_tsne = rng.choice(len(X_pca_vis), size=min(max_tsne, len(X_pca_vis)), replace=False)
X_tsne_in = X_pca_vis[idx_tsne]
y_tsne = y[idx_tsne]

if len(X_tsne_in) > 10:
    tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto', init='pca', random_state=RANDOM_STATE)
    X_tsne = tsne.fit_transform(X_tsne_in)

    plt.figure(figsize=(7, 5))
    palette = {0: 'blue', 1: 'red'}
    for cls in np.unique(y_tsne):
        mask = y_tsne == cls
        plt.scatter(X_tsne[mask, 0], X_tsne[mask, 1], s=18, alpha=0.7, label='NORMAL' if cls == 0 else 'PNEUMONIA', c=palette[cls])
    plt.title('t-SNE (amostra)')
    plt.legend()
    plt.tight_layout()
    plt.show()

### **CELULA 6: VALIDACAO CRUZADA ESTRATIFICADA E AVALIACAO**

Nesta etapa, usamos validacao cruzada estratificada para comparar SVM (com diferentes kernels) e Random Forest.
As metricas incluem acuracia, sensibilidade, especificidade, F1, AUC e tempo de processamento.

In [ ]:
# ==============================================================================
# CELULA 6: VALIDACAO CRUZADA ESTRATIFICADA E AVALIACAO
# ==============================================================================
N_SPLITS = 5
USAR_PCA_NO_TREINO = True
PCA_COMPONENTES = 0.95

def calcular_metricas(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    acc = accuracy_score(y_true, y_pred)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1 = f1_score(y_true, y_pred)
    return {
        'Acuracia': acc,
        'Sensibilidade': sens,
        'Especificidade': spec,
        'F1': f1
    }

def criar_pipeline(modelo):
    etapas = [('scaler', StandardScaler())]
    if USAR_PCA_NO_TREINO:
        etapas.append(('pca', PCA(n_components=PCA_COMPONENTES, random_state=RANDOM_STATE)))
    etapas.append(('clf', modelo))
    return Pipeline(etapas)

def avaliar_modelo_cv(modelo, X_data, y_data, score_method='auto', n_splits=N_SPLITS):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    metricas = []
    y_true_all = []
    y_score_all = []
    fit_times = []
    pred_times = []

    for train_idx, test_idx in skf.split(X_data, y_data):
        X_train, X_test = X_data[train_idx], X_data[test_idx]
        y_train, y_test = y_data[train_idx], y_data[test_idx]

        clf = clone(modelo)
        t0 = time.perf_counter()
        clf.fit(X_train, y_train)
        fit_times.append(time.perf_counter() - t0)

        t1 = time.perf_counter()
        y_pred = clf.predict(X_test)
        pred_times.append(time.perf_counter() - t1)

        if score_method == 'auto':
            if hasattr(clf, 'decision_function'):
                y_score = clf.decision_function(X_test)
            else:
                y_score = clf.predict_proba(X_test)[:, 1]
        elif score_method == 'proba':
            y_score = clf.predict_proba(X_test)[:, 1]
        else:
            y_score = clf.decision_function(X_test)

        metricas.append(calcular_metricas(y_test, y_pred))
        y_true_all.append(y_test)
        y_score_all.append(y_score)

    df_metricas = pd.DataFrame(metricas)
    y_true_all = np.concatenate(y_true_all)
    y_score_all = np.concatenate(y_score_all)
    auc = roc_auc_score(y_true_all, y_score_all)
    fpr, tpr, _ = roc_curve(y_true_all, y_score_all)

    return {
        'metricas_mean': df_metricas.mean(),
        'metricas_std': df_metricas.std(),
        'auc': auc,
        'fpr': fpr,
        'tpr': tpr,
        'fit_time_mean': float(np.mean(fit_times)),
        'pred_time_mean': float(np.mean(pred_times))
    }

svm_linear = criar_pipeline(SVC(kernel='linear', C=1.0, class_weight='balanced', random_state=RANDOM_STATE))
svm_rbf = criar_pipeline(SVC(kernel='rbf', C=1.0, gamma='scale', class_weight='balanced', random_state=RANDOM_STATE))
svm_poly = criar_pipeline(SVC(kernel='poly', degree=3, C=1.0, gamma='scale', class_weight='balanced', random_state=RANDOM_STATE))

rf_base = criar_pipeline(RandomForestClassifier(
    n_estimators=200,
    max_depth=None,
    class_weight='balanced',
    n_jobs=-1,
    random_state=RANDOM_STATE
))

modelos = {
    'SVM Linear': svm_linear,
    'SVM RBF': svm_rbf,
    'SVM Polinomial': svm_poly,
    'Random Forest (200)': rf_base
}

resultados = {}
for nome, modelo in modelos.items():
    print(f'\nAvaliando: {nome}')
    resultados[nome] = avaliar_modelo_cv(modelo, X, y)
    print(f"AUC: {resultados[nome]['auc']:.4f}")

### **CELULA 7: COMPARACAO DE METRICAS**

Consolida as metricas e compara os modelos em tabela e grafico.

In [ ]:
# ==============================================================================
# CELULA 7: COMPARACAO DE METRICAS
# ==============================================================================
linhas = []
for nome, res in resultados.items():
    linhas.append({
        'Modelo': nome,
        'Acuracia': res['metricas_mean']['Acuracia'],
        'Sensibilidade': res['metricas_mean']['Sensibilidade'],
        'Especificidade': res['metricas_mean']['Especificidade'],
        'F1': res['metricas_mean']['F1'],
        'AUC': res['auc'],
        'Tempo Treino (s)': res['fit_time_mean'],
        'Tempo Pred (s)': res['pred_time_mean']
    })

df_res = pd.DataFrame(linhas).sort_values('AUC', ascending=False)
print(df_res)

metricas_plot = df_res.melt(
    id_vars='Modelo',
    value_vars=['Acuracia', 'Sensibilidade', 'Especificidade', 'F1', 'AUC'],
    var_name='Metrica',
    value_name='Valor'
)

plt.figure(figsize=(11, 6))
sns.barplot(data=metricas_plot, x='Metrica', y='Valor', hue='Modelo')
plt.ylim(0, 1.1)
plt.title('Comparacao de Metricas')
plt.tight_layout()
plt.show()

### **CELULA 8: CURVA ROC E AUC (COMPARATIVO)**

Plota as curvas ROC dos modelos avaliados.

In [ ]:
# ==============================================================================
# CELULA 8: CURVA ROC E AUC
# ==============================================================================
plt.figure(figsize=(8, 6))
for nome, res in resultados.items():
    plt.plot(res['fpr'], res['tpr'], label=f"{nome} (AUC={res['auc']:.3f})")

plt.plot([0, 1], [0, 1], 'k--', label='Aleatorio')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('Curva ROC - Comparacao')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### **CELULA 9: IMPACTO DE ARVORES E PROFUNDIDADE NO RANDOM FOREST**

Varia o numero de arvores e a profundidade para observar impacto em F1 e AUC.

In [ ]:
# ==============================================================================
# CELULA 9: IMPACTO DE ARVORES E PROFUNDIDADE NO RANDOM FOREST
# ==============================================================================
N_SPLITS_RF = 3
n_estimators_list = [100, 200, 400]
max_depth_list = [None, 10, 20]

result_grid = []
for n_est in n_estimators_list:
    for depth in max_depth_list:
        rf = criar_pipeline(RandomForestClassifier(
            n_estimators=n_est,
            max_depth=depth,
            class_weight='balanced',
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
        res = avaliar_modelo_cv(rf, X, y, score_method='proba', n_splits=N_SPLITS_RF)
        result_grid.append({
            'n_estimators': n_est,
            'max_depth': 'None' if depth is None else depth,
            'F1': res['metricas_mean']['F1'],
            'AUC': res['auc']
        })

rf_grid_df = pd.DataFrame(result_grid)
print(rf_grid_df)

pivot_auc = rf_grid_df.pivot(index='max_depth', columns='n_estimators', values='AUC')
plt.figure(figsize=(7, 4))
sns.heatmap(pivot_auc, annot=True, fmt='.3f', cmap='viridis')
plt.title('AUC por n_estimators e max_depth')
plt.tight_layout()
plt.show()

### **CELULA 10: RESUMO DE TEMPO DE PROCESSAMENTO**

Resumo do tempo medio de treino e predicao por modelo.

In [ ]:
# ==============================================================================
# CELULA 10: RESUMO DE TEMPO
# ==============================================================================
df_time = df_res[['Modelo', 'Tempo Treino (s)', 'Tempo Pred (s)']].copy()
print(df_time)

plt.figure(figsize=(8, 4))
sns.barplot(
    data=df_time.melt(id_vars='Modelo', var_name='Tipo', value_name='Tempo'),
    x='Modelo',
    y='Tempo',
    hue='Tipo'
)
plt.title('Tempo medio por modelo')
plt.tight_layout()
plt.show()